# Exploratory Data Analysis

Before doing any real analysis, I always want to understand what I'm actually working with. How big is the data, what's missing, what does the distribution look like — the basics that tell you whether your data is trustworthy or whether you're going to hit surprises halfway through.

This dataset has five tables totalling just over 2 million rows, which is a decent size for local analysis. DuckDB handles it without breaking a sweat.


In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.ingest import load_all
from pipeline.clean import clean_all
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

raw = load_all()
cleaned = clean_all(raw)


## How big is everything?

First thing — let's just confirm what we're working with row-count wise. The events table is by far the largest, which makes sense — every click, view, bounce and purchase gets logged.


In [ ]:

for name, df in cleaned.items():
    print(f"{name:15s}  rows={len(df):>10,}  cols={df.shape[1]}")


## Nulls — what's missing and does it matter?

Nulls in events aren't surprising — a bounce event won't have a product_id because the user left before viewing one. Same logic applies to transactions. The key question is whether the nulls are random or systematic. If all the nulls are in bounce events, that's fine. If they're scattered across purchase events, that's a problem.


In [ ]:

for name, df in cleaned.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n--- {name} ---")
        print((nulls / len(df) * 100).round(2).to_string())
    else:
        print(f"{name}: no nulls")


## Customer age distribution

Who are the customers? The age distribution tells you whether this is a young-skewing platform or a broader demographic. Worth knowing because it affects how you'd interpret campaign performance across channels like Email vs Social.


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
cleaned['customers']['age'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Customer Age Distribution')
ax.set_xlabel('Age'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()


## Loyalty tier split

The loyalty tier breakdown matters because it tells you how concentrated your customer base is. If 80% are Bronze, your retention strategy needs to be very different than if most customers are already Gold or Platinum.


In [ ]:

tier_counts = cleaned['customers']['loyalty_tier'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
tier_counts.plot(kind='bar', ax=ax, color=['#cd7f32','#c0c0c0','#ffd700'], edgecolor='white')
ax.set_title('Customer Loyalty Tier Distribution')
ax.set_xlabel('Tier'); ax.set_ylabel('Count')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## What events are users generating?

This gives a quick read on user behaviour. A high bounce count relative to views is expected — most people don't engage. What you're looking for is whether the purchase count looks proportionate given the view count. If it's tiny, something's broken in the funnel.


In [ ]:

ev_counts = cleaned['events']['event_type'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
ev_counts.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Event Type Counts'); ax.set_xlabel('Count')
plt.tight_layout(); plt.show()


## Revenue distribution

This is one of the more important EDA checks. Revenue distributions in e-commerce are almost always right-skewed — a few large orders pull the mean up. The negative values are refunds, which is expected. Worth flagging that we should use median AOV alongside mean AOV when reporting.


In [ ]:

tx = cleaned['transactions'].dropna(subset=['gross_revenue'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tx['gross_revenue'].clip(-200, 600).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Gross Revenue Distribution'); axes[0].set_xlabel('Revenue ($)')
tx['quantity'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Quantity per Transaction'); axes[1].set_xlabel('Quantity')
plt.tight_layout(); plt.show()


## Product categories

Just a sanity check on how the product catalogue breaks down. Useful context before we go deeper into product performance — if Sports has 5 products and Grocery has 500, raw revenue comparisons between categories are misleading without normalisation.


In [ ]:

cat_counts = cleaned['products']['category'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
cat_counts.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Products by Category'); ax.set_xlabel('Category'); ax.set_ylabel('Count')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## Monthly trends — the big picture

Before drilling into anything specific, it's worth looking at whether the business is growing, declining, or flat. Seasonality also shows up here, which matters if you're benchmarking campaign performance across months.


In [ ]:

monthly = cleaned['transactions'].groupby('month').agg(
    orders=('transaction_id', 'count'),
    revenue=('net_revenue', 'sum')
).reset_index()
fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.bar(monthly['month'], monthly['orders'], color='steelblue', alpha=0.6, label='Orders')
ax2.plot(monthly['month'], monthly['revenue'], color='crimson', marker='o', lw=2, label='Revenue')
ax1.set_title('Monthly Orders & Revenue')
ax1.set_xlabel('Month'); ax1.set_ylabel('Orders'); ax2.set_ylabel('Net Revenue ($)')
plt.xticks(rotation=45, ha='right')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout(); plt.show()
